In [18]:
import numpy as np
import pandas as pd
import helper

In [19]:
distribution_name = 'exponential'
# algos = ['km', 'joint_order_saa']
algos = ['km']



In [20]:
qbar, param_list, b_list, rho_list, lam_list = helper.get_parameters(distribution_name)

demand_list = helper.sample_dist(int(1e7), distribution_name, param_list[-1])

qopt = helper.get_optimal_quantile(9, 1, demand_list)
print(f'qopt value: {qopt}')
lam_list = np.linspace(0.95*qopt, 1.05*qopt, 4)


file_loc = f'./data/{distribution_name}_knife_edge.csv'


qopt value: 184.27069331927316
qopt value: 184.20259947501268


In [21]:
df = pd.read_csv(file_loc)

In [22]:
print(df['algorithm'].unique())

['ignorant_saa' 'subsample_saa' 'robust' 'km' 'joint_order_saa'
 'robust_plus' 'true_saa']


In [23]:

algo_df = df[df['algorithm'].isin(algos)]

In [24]:
print(algo_df)

      algorithm       metric                value    N  b  h distribution  \
12           km    true_cost   184.60623407662226  100  9  1  exponential   
13           km  minmax_cost    18.36977470357779  100  9  1  exponential   
14           km           id         identifiable  100  9  1  exponential   
15           km     estimate   174.96304403497683  100  9  1  exponential   
39           km    true_cost   185.31033920767916  100  9  1  exponential   
...         ...          ...                  ...  ... .. ..          ...   
32361        km     estimate   191.51104486778084  500  9  1  exponential   
32385        km    true_cost    184.1565223315486  500  9  1  exponential   
32386        km  minmax_cost  0.10117056741904662  500  9  1  exponential   
32387        km           id         identifiable  500  9  1  exponential   
32388        km     estimate    188.1892464333879  500  9  1  exponential   

       param         lam  order_level_two  qbar  
12        80  174.963044 

In [25]:
algo_df.columns

Index(['algorithm', 'metric', 'value', 'N', 'b', 'h', 'distribution', 'param',
       'lam', 'order_level_two', 'qbar'],
      dtype='object')

In [26]:
# For each value of lambda, compute:
# \qcrit_G if unidentifiable
# Delta

lam_vals = algo_df['lam'].unique()
print(f'Lambda values: {lam_vals}')



Lambda values: [174.96304403 181.10209821 187.24115239 193.38020656]


In [27]:
delta_data = {}
b = 9
h = 1
for lam in lam_vals:
    qcrit = helper.get_optimal_robust(b, h, lam, qbar, demand_list) 
    risk = helper.get_robust_cost(qcrit, b, h, lam, qbar, demand_list)
    delta_data[lam] = (qcrit, risk)


In [28]:
print(delta_data)

{174.96304403497683: (191.29176153303902, 16.32871749806219), 181.1020982116427: (186.61441136378912, 5.512313152146419), 187.24115238830856: (184.20259947501268, 0.0), 193.38020656497443: (184.20259947501268, 0.0)}


In [29]:
# Round lam values in both delta_data and algo_df for safe matching
lam_round = 6  # number of decimals to round to, adjust if needed

# Create a DataFrame from delta_data for merging
delta_df = pd.DataFrame(
    [(round(lam, lam_round), qcrit, delta) for lam, (qcrit, delta) in delta_data.items()],
    columns=['lam', 'qcrit', 'Delta']
)

# Round lam in algo_df for merge
algo_df['lam_rounded'] = algo_df['lam'].round(lam_round)

# Merge on rounded lambda
algo_df = algo_df.merge(delta_df, left_on='lam_rounded', right_on='lam', how='left')

# Drop helper columns if desired
algo_df = algo_df.drop(['lam_rounded', 'lam_y'], axis=1).rename(columns={'lam_x': 'lam'})

/var/folders/8g/9lq_cd3s6314xyx0gyq0f2qh0000gn/T/ipykernel_20655/1699795100.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  algo_df['lam_rounded'] = algo_df['lam'].round(lam_round)


In [30]:
algo_df

,algorithm,metric,value,N,b,h,distribution,param,lam,order_level_two,qbar,qcrit,Delta
0,km,true_cost,184.60623407662226,100,9,1,exponential,80,174.963044,101.0,325,191.291762,16.328717
1,km,minmax_cost,18.36977470357779,100,9,1,exponential,80,174.963044,101.0,325,191.291762,16.328717
2,km,id,identifiable,100,9,1,exponential,80,174.963044,101.0,325,191.291762,16.328717
3,km,estimate,174.96304403497683,100,9,1,exponential,80,174.963044,101.0,325,191.291762,16.328717
4,km,true_cost,185.31033920767916,100,9,1,exponential,80,174.963044,109.0,325,191.291762,16.328717
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4795,km,estimate,191.51104486778084,500,9,1,exponential,80,193.380207,67.0,325,184.202599,0.000000
4796,km,true_cost,184.1565223315486,500,9,1,exponential,80,193.380207,59.0,325,184.202599,0.000000
4797,km,minmax_cost,0.10117056741904662,500,9,1,exponential,80,193.380207,59.0,325,184.202599,0.000000
4798,km,id,identifiable,500,9,1,exponential,80,193.380207,59.0,325,184.202599,0.000000


In [31]:
wide = (
    algo_df
    .assign(rep=lambda df: df.groupby(
        ["algorithm","N","lam","Delta","metric"]
    ).cumcount())
    .pivot(
        index=["algorithm","N","lam","Delta","rep"],
        columns="metric",
        values="value"
    )
    .reset_index()
)

numeric_cols = ["true_cost", "minmax_cost", "estimate", "Delta"]

for c in numeric_cols:
    wide[c] = pd.to_numeric(wide[c], errors="coerce")


wide["minmax_cost"] = pd.to_numeric(wide["minmax_cost"])
wide["Delta"] = pd.to_numeric(wide["Delta"])

wide["minmax_cost"] = wide["minmax_cost"] - wide["Delta"]

print(wide.head(5))


metric algorithm    N         lam      Delta  rep    estimate            id  \
0             km  100  174.963044  16.328717    0  174.963044  identifiable   
1             km  100  174.963044  16.328717    1  170.425845  identifiable   
2             km  100  174.963044  16.328717    2  174.963044  identifiable   
3             km  100  174.963044  16.328717    3  174.470951  identifiable   
4             km  100  174.963044  16.328717    4  174.963044  identifiable   

metric  minmax_cost   true_cost  
0          2.041057  184.606234  
1          2.745162  185.310339  
2          2.041057  184.606234  
3          2.103047  184.668224  
4          2.041057  184.606234  


In [32]:
def stderr(x):
    return x.std(ddof=1) / np.sqrt(len(x))

summary = (
    wide
    .groupby(["algorithm", "N", "lam"])
    .agg(
        true_cost_mean=("true_cost", "mean"),
        true_cost_stderr=("true_cost", stderr),

        minmax_cost_mean=("minmax_cost", "mean"),
        minmax_cost_stderr=("minmax_cost", stderr),

        estimate_mean=("estimate", "mean"),
        estimate_stderr=("estimate", stderr),
    )
    .reset_index()
)

In [33]:
summary

,algorithm,N,lam,true_cost_mean,true_cost_stderr,minmax_cost_mean,minmax_cost_stderr,estimate_mean,estimate_stderr
0,km,100,174.963044,185.702047,0.270471,3.136870,0.270471,170.969504,0.788262
1,km,100,181.102098,185.710231,0.330175,1.708482,0.330175,173.565387,1.096586
2,km,100,187.241152,185.438999,0.299204,1.383647,0.299204,177.813771,1.246697
3,km,100,193.380207,186.262270,0.362172,2.206918,0.362172,177.829214,1.653629
4,km,250,174.963044,184.927475,0.100342,2.362298,0.100342,173.420760,0.391819
5,km,250,181.102098,184.639302,0.098541,0.637552,0.098541,177.026211,0.601711
6,km,250,187.241152,184.674902,0.115119,0.619550,0.115119,180.155414,0.872438
7,km,250,193.380207,184.847218,0.114516,0.791866,0.114516,181.607246,1.066006
8,km,500,174.963044,184.790089,0.046071,2.224912,0.046071,173.904691,0.252543
9,km,500,181.102098,184.433582,0.058494,0.431832,0.058494,178.186221,0.465479


In [34]:
id_pct = (
    wide
    .groupby(["algorithm", "N", "lam"])["id"]
    .value_counts(normalize=True)
    .unstack(fill_value=0)
    .reset_index()
)

print(id_pct)

id algorithm    N         lam  identifiable
0         km  100  174.963044           1.0
1         km  100  181.102098           1.0
2         km  100  187.241152           1.0
3         km  100  193.380207           1.0
4         km  250  174.963044           1.0
5         km  250  181.102098           1.0
6         km  250  187.241152           1.0
7         km  250  193.380207           1.0
8         km  500  174.963044           1.0
9         km  500  181.102098           1.0
10        km  500  187.241152           1.0
11        km  500  193.380207           1.0
